# UC2 — Metal Surface (Magnetic Tile): 4. Synthetic Data Generation

Generate synthetic Metal Surface anomaly images from the testcase built in notebook 3,
using the released (or your own) AnomalyGen checkpoint. The model inpaints a learned
defect into each masked region of a clean image.

> **How commands run in this tutorial.** All pipeline steps run inside the
> `cosmos-predict2` conda environment. In a notebook cell we prefix shell
> commands with `conda run -n cosmos-predict2` (add `--live-stream` to stream
> logs live). If you prefer, open a JupyterLab **Terminal**, run
> `conda activate cosmos-predict2` once, and paste the same commands without
> the `conda run` prefix.
>
> If that environment does not exist yet, build it first with the top-level
> [tutorial/notebooks/0-setup-cuda128.ipynb](../../0-setup-cuda128.ipynb) — see
> the prerequisite note below.

## 4.0 Set the project root

In [ ]:
# Resolve the repository root (the folder containing pyproject.toml) and cd into it,
# so every relative path below (datasets/, checkpoints/, results/, scripts/) resolves.
import os
d = os.getcwd()
while d != "/" and not os.path.exists(os.path.join(d, "pyproject.toml")):
    d = os.path.dirname(d)
LOCAL_PROJECT_DIR = d
os.chdir(LOCAL_PROJECT_DIR)
# Pipeline scripts read the finetuned models & write outputs under the repo root.
os.environ.setdefault("IMAGINAIRE_OUTPUT_ROOT", "./results")
print("Project root:", LOCAL_PROJECT_DIR)

## 4.1 Generation configuration

Each line of `testcase.jsonl` drives one generation. Key fields:

| Field | Meaning |
|---|---|
| `image_filename` | clean canvas image |
| `mask_filename` | defect mask (must match the image size) |
| `anomaly_type` | `TEXTURE+TYPE`, must be supported by the checkpoint |
| `guidance` | strength of the anomaly condition (default 7.0 here) |
| `num_steps` | denoising steps (default 35) |
| `crop_and_paste` / `crop_ratio` | crop around the mask, generate, paste back |

> **Mask size must match the image size** — AMP already sized the masks to their clean
> canvases in notebook 3, so the testcase is generation-ready.

## 4.2 Run generation

This loads the diffusion backbone once and generates every entry in the testcase into
`results/UC2_metal/example_output/`.

> **Using your own checkpoint?** The command below points at the released Metal checkpoint
> (`--ag_checkpoint_dir=checkpoints/nvidia/Cosmos-AnomalyGen-Metal-2B --step=10000`). If you
> self-trained (notebook 2), repoint both: set `--ag_checkpoint_dir` to your run's
> `results/anomaly_gen/<group>/<name>` folder and `--step` to the checkpoint you picked (§2.4).

In [ ]:
!conda run -n cosmos-predict2 bash -c "export IMAGINAIRE_OUTPUT_ROOT=./results && \
 CUDA_VISIBLE_DEVICES=0 torchrun --nproc_per_node=1 --master_port=12360 \
   -m scripts.anomaly_gen.synthetic_dataset_generation \
   --config=cosmos_predict2/configs/base/ag_config.py \
   --ag_checkpoint_dir=checkpoints/nvidia/Cosmos-AnomalyGen-Metal-2B \
   --step=10000 \
   --input_data_path=ag_inference/UC2_metal/testcase.jsonl \
   --output_image_path=results/UC2_metal/example_output \
   --seed=0 \
   -- experiment=predict2_anomaly_gen_ddp_2b"

## 4.3 Output structure

```
results/UC2_metal/example_output/
├── original_image/        # input clean images
├── original_mask/         # input (AMP-placed) masks
├── cropped_image/         # crops around the masked region
├── cropped_mask/          # cropped masks
├── annotated_image/       # clean image with the crop region drawn
├── reconstructed_image/   # final outputs with the inpainted defect
└── SDG_result.csv         # per-sample metadata (paths, guidance, seed, PSNR, ...)
```

#### Example results

Each row: the clean canvas, the placed defect mask, and the AnomalyGen reconstruction
(background preserved, a learned defect painted inside the masked region). These were
produced by the generation cell above.

| Defect | Clean image | Input mask | Reconstructed |
|---|---|---|---|
| metal_surface+MT_Blowhole | ![](assets/generation/metal_surface+MT_Blowhole_clean.png) | ![](assets/generation/metal_surface+MT_Blowhole_mask.png) | ![](assets/generation/metal_surface+MT_Blowhole_recon.png) |
| metal_surface+MT_Break | ![](assets/generation/metal_surface+MT_Break_clean.png) | ![](assets/generation/metal_surface+MT_Break_mask.png) | ![](assets/generation/metal_surface+MT_Break_recon.png) |
| metal_surface+MT_Crack | ![](assets/generation/metal_surface+MT_Crack_clean.png) | ![](assets/generation/metal_surface+MT_Crack_mask.png) | ![](assets/generation/metal_surface+MT_Crack_recon.png) |
| metal_surface+MT_Fray | ![](assets/generation/metal_surface+MT_Fray_clean.png) | ![](assets/generation/metal_surface+MT_Fray_mask.png) | ![](assets/generation/metal_surface+MT_Fray_recon.png) |
| metal_surface+MT_Uneven | ![](assets/generation/metal_surface+MT_Uneven_clean.png) | ![](assets/generation/metal_surface+MT_Uneven_mask.png) | ![](assets/generation/metal_surface+MT_Uneven_recon.png) |


## 4.4 Evaluate

Compare generated defects against the real ones. **nn_score** (nearest-neighbor
DINOv2 correspondence) is the primary KPI (higher = more like real defects);
**mnn_score** is the mutual variant; **FID** is distribution-level (lower is better).

In [ ]:
!conda run -n cosmos-predict2 python -m scripts.anomaly_gen.evaluate \
    --real_path datasets/UC2_metal \
    --generated_path results/UC2_metal/example_output \
    --anomaly_types metal_surface+MT_Blowhole metal_surface+MT_Break metal_surface+MT_Crack metal_surface+MT_Fray metal_surface+MT_Uneven

> **FID note.** FID needs **more than two** samples per anomaly type in *both* the
> real and generated sets; with a tiny testcase it may report `None`. Increase
> `--num-sdg` in notebook 3 to get enough samples for a meaningful FID.

## 4.5 (Optional) Filter by quality

Split generated samples into `keep/` and `drop/` by per-sample G-IQA score.
`--drop_ratio 0.2` discards the lowest-scoring 20% per defect type.

In [ ]:
!conda run -n cosmos-predict2 python -m scripts.anomaly_gen.filter \
    --real_path datasets/UC2_metal \
    --generated_path results/UC2_metal/example_output \
    --output_path results/UC2_metal/filtered \
    --drop_ratio 0.2 \
    --anomaly_types metal_surface+MT_Blowhole metal_surface+MT_Break metal_surface+MT_Crack metal_surface+MT_Fray metal_surface+MT_Uneven

## Next Step

Proceed to [5-pseudo-labeling.ipynb](./5-pseudo-labeling.ipynb).